In [7]:
from datetime import datetime
from openpyxl import Workbook
from openpyxl.styles import Font, Alignment, Border, Side


def create_invoice():
    # ===== 日付設定 =====
    today = datetime.today()
    cell_date = today.strftime("%Y/%m/%d")   # 例：2023/06/06
    file_date = today.strftime("%Y%m%d")     # 例：20230520
    filename = f"請求書_{file_date}.xlsx"

    # ===== Workbook / Worksheet =====
    wb = Workbook()
    ws = wb.active
    ws.title = "請求書"

    # ===== スタイル定義 =====
    bold_font = Font(bold=True)
    title_font = Font(bold=True, size=16)

    thin = Side(style="thin")
    border_thin = Border(left=thin, right=thin, top=thin, bottom=thin)

    left = Alignment(horizontal="left", vertical="center")
    center = Alignment(horizontal="center", vertical="center")
    right = Alignment(horizontal="right", vertical="center")

    # ===== 列幅 =====
    ws.column_dimensions["A"].width = 2
    ws.column_dimensions["B"].width = 18
    ws.column_dimensions["C"].width = 10
    ws.column_dimensions["D"].width = 12
    ws.column_dimensions["E"].width = 14
    ws.column_dimensions["F"].width = 6
    ws.column_dimensions["G"].width = 14

    # ===== タイトル =====
    ws["B2"] = "請求書"
    ws["B2"].font = title_font
    ws["B2"].alignment = left

    # ===== 請求元情報 =====
    ws["B4"] = "株式会社ABC"
    ws["B5"] = "〒101-0022 東京都千代田区神田練塀町300"
    ws["B6"] = "TEL:03-1234-5678  FAX:03-1234-5678"
    ws["B7"] = "担当者名:鈴木一郎 様"

    for addr in ["B4", "B5", "B6", "B7"]:
        ws[addr].font = bold_font
        ws[addr].alignment = left

    # ===== 右上（No. / 日付）=====
    ws["F4"] = "No."
    ws["G4"] = "0001"
    ws["F5"] = "日付"
    ws["G5"] = cell_date

    for addr in ["F4", "F5", "G4", "G5"]:
        ws[addr].font = bold_font
        ws[addr].alignment = left

    # ===== 明細ヘッダ =====
    headers = ["商品名", "数量", "単価", "金額"]
    cols = ["B", "C", "D", "E"]
    for col, header in zip(cols, headers):
        cell = ws[f"{col}10"]
        cell.value = header
        cell.font = bold_font
        cell.alignment = center
        cell.border = border_thin

    # ===== 明細 =====
    items = [
        ("商品A", 2, 10000, 20000),
        ("商品B", 1, 15000, 15000),
    ]

    start_row = 11
    for i, (name, qty, price, amount) in enumerate(items):
        r = start_row + i

        ws[f"B{r}"] = name
        ws[f"C{r}"] = qty
        ws[f"D{r}"] = price
        ws[f"E{r}"] = amount

        ws[f"B{r}"].font = bold_font
        ws[f"B{r}"].alignment = left

        for c in ["C", "D", "E"]:
            ws[f"{c}{r}"].font = bold_font
            ws[f"{c}{r}"].alignment = right
            ws[f"{c}{r}"].number_format = "#,##0"

        for c in ["B", "C", "D", "E"]:
            ws[f"{c}{r}"].border = border_thin

    # ===== 金額計算 =====
    subtotal = sum(item[3] for item in items)
    tax = int(subtotal * 0.10)
    total = subtotal + tax

    # ===== 小計・消費税・合計 =====
    ws["B15"] = "小計"
    ws["B16"] = "消費税"
    ws["B17"] = "合計"

    for r in [15, 16, 17]:
        ws[f"B{r}"].font = bold_font
        ws[f"B{r}"].alignment = left
        ws[f"B{r}"].border = border_thin

        # C〜Eを結合
        ws.merge_cells(start_row=r, start_column=3, end_row=r, end_column=5)

    # 金額を結合セル（C列）に設定
    ws["C15"] = subtotal
    ws["C16"] = tax
    ws["C17"] = total

    # ===== 結合セル全体に罫線を設定 =====
    for r in [15, 16, 17]:
        for c in range(3, 6):  # C(3) ～ E(5)
            cell = ws.cell(row=r, column=c)
            cell.border = border_thin

        cell = ws[f"C{r}"]
        cell.font = bold_font
        cell.alignment = right
        cell.number_format = "#,##0"

    # ===== 保存 =====
    wb.save(filename)
    print(f"作成しました: {filename}")


if __name__ == "__main__":
    create_invoice()

作成しました: 請求書_20260301.xlsx
